# Producto U1 consolidado — Equipo LLSW3 (sección GU)

**Proyecto:** análisis de tráfico de red del campus universitario (UPeU Juliaca) — dataset propio, capturado vía Suricata (`TRCU.csv`, 397 354 flujos reales, 83 columnas).

**Integrantes y dimensión U1 de cada uno:**

| Integrante | Dimensión | Tipo de modelo | Variable objetivo |
|---|---|---|---|
| Nick Saim Mayta Jara | Volumen de tráfico por flujo | Regresión | `bytes_per_s` |
| Jhan Logan Ramos Quispe | Duración del flujo de red | Regresión | `flow_duration` |
| Henyelrey Lucio Garcia Chura | Tipo de servicio vs. catálogo IANA | Clasificación | `categoria_servicio_ref` |
| David Romero Nina | Dirección dominante del flujo | Clasificación | `categoria_direccion` (de `down_up_ratio`) |

**Metodología:** CRISP-DM — Fases 1 a 5 (hasta modelado/evaluación; el despliegue en streaming es Unidad 2). Este notebook **consolida en un solo documento ejecutable** las cuatro dimensiones individuales (`u1_producto_nick.ipynb`, `u1_producto_jhan.ipynb`, `u1_producto_henyelrey.ipynb`, `u1_producto_david.ipynb`), reutilizando una sola extracción del dataset (Fase 2 general) y desarrollando cada dimensión como su propio bloque Fase 3 → 4 → 5, para que se lea como el proceso CRISP-DM completo del equipo sobre el mismo instrumento.

Contexto completo del proyecto: [Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2).

## Glosario general

| Término | Significado |
|---|---|
| `flow_id` | Identificador único del flujo (5-tupla: IP/puerto origen y destino + protocolo). |
| `bytes_per_s` | Tasa de transferencia del flujo — indicador directo de carga sobre la red (dimensión de Nick). |
| `flow_duration` | Duración total del flujo, en microsegundos (dimensión de Jhan). |
| `dst_port` | Puerto de destino — fuente de la etiqueta de referencia de servicio (dimensión de Henyelrey), excluido como predictor para evitar fuga de información. |
| `down_up_ratio` | Proporción entre tráfico de bajada y de subida — valores altos indican descarga dominante (dimensión de David). |
| IAT (*inter-arrival time*) | Tiempo entre paquetes consecutivos del mismo flujo (`iat_*`, `fwd_iat_*`, `bwd_iat_*`). |
| `active_*` / `idle_*` | Estadísticas de los períodos en que el flujo estuvo activo o inactivo. |
| Puertos IANA (*well-known ports*) | Catálogo oficial puerto→servicio (iana.org/assignments/service-names-port-numbers); aquí se usa una versión simplificada de 29 puertos. |
| Fuga de información (*data leakage*) | Usar una columna que es, de hecho, la fuente de la propia etiqueta como predictor — se excluye a propósito en las dimensiones de Henyelrey y David. |
| Línea base ingenua | Predictor trivial (siempre el promedio/mediana, o siempre la clase mayoritaria) usado como piso de comparación. |
| Criterio de éxito | Umbral definido *antes* de entrenar que dice cuándo un modelo aporta valor real para la decisión de negocio que habilita. |
| CRISP-DM | Metodología estándar de minería de datos en 6 fases: comprensión del negocio, de los datos, preparación, modelado, evaluación y despliegue. Este notebook cubre las primeras 5; el despliegue (inferencia en streaming) es contenido de Unidad 2. |

## Arquitectura Big Data (contexto — no es una fase de CRISP-DM)

Este notebook implementa la **ruta batch** de la arquitectura **Lambda** declarada en el [Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2): capa batch (este notebook, sobre el histórico `TRCU.csv`) + capa de velocidad (Kafka + Spark Structured Streaming, contenido de Unidad 2 — dimensión U2 de cada integrante). La razón de Lambda sobre Kappa: la columna `label` del dataset está sin asignar (`NeedLabel` en todos los registros) y su validación es por lotes, no dato por dato — los modelos U1 necesitan recomputar sobre el histórico completo cada vez que el criterio de etiquetado se ajusta.

## Fase 1 — Comprensión del negocio (CRISP-DM)

**Pregunta central del Proyecto Sello:** ¿Cómo caracterizar el comportamiento de los flujos de tráfico de red del campus — volumen, duración, tipo de servicio y dirección dominante — para anticipar la carga esperada de la red y detectar patrones que se aparten de lo habitual, apoyando la gestión de capacidad y la vigilancia del equipo de TI?

Cada integrante aporta una dimensión propia de esa pregunta central:

| Integrante | Pregunta de negocio (dimensión propia) | Objetivo de minería de datos | Decisión que habilita | Criterio de éxito |
|---|---|---|---|---|
| Nick | ¿Cuál ha sido el volumen histórico de tráfico (bytes/s), y qué volumen se puede esperar? | Regresión de `bytes_per_s` | Anticipar picos de carga y priorizar capacidad de red | RMSE bajo línea base; R² > 0.5 |
| Jhan | ¿Qué duración histórica han tenido los flujos según sus características iniciales? | Regresión de `flow_duration` | Dimensionar ventanas de sesión y retención en Kafka (U2) | Reducción de RMSE ≥ 20% vs. línea base |
| Henyelrey | ¿Qué tipo de servicio corresponde a cada flujo según su comportamiento, vs. el catálogo IANA? | Clasificación de `categoria_servicio_ref` (sin usar el puerto) | Caracterizar la composición del tráfico por servicio | +15-20 pp de accuracy vs. línea base |
| David | ¿Qué proporción de flujos son de descarga dominante, carga dominante o balanceados? | Clasificación de `categoria_direccion` (3 categorías por cuantiles) | Priorizar ancho de banda saliente vs. entrante | Accuracy > 50% (línea base ~33-34% por diseño) |

Detalle completo de cada ficha en el [Brief técnico-analítico](../../docs/proyecto-sello/brief.md), sección 3.

## Fase 2 — Comprensión de los datos (CRISP-DM)

### Extracción con esquema explícito (una sola vez para las 4 dimensiones)

Hallazgo documentado en S03: con `header=True` + `StructType` explícito, Spark asigna los campos **por posición**, no por nombre. Si el orden de `StructField` no coincide con el orden físico del CSV, los valores se corrompen en silencio. Por eso el esquema de abajo respeta el header real observado en `TRCU.csv`, columna por columna, y se valida con un `assert` antes de continuar.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)
from pyspark.sql import functions as F

# Este notebook entrena 4 dimensiones (8 modelos MLlib en total) en una sola sesion de
# Spark, a diferencia de los notebooks individuales (una dimension por proceso/JVM) —
# por eso el driver necesita mas heap que el default (~1g) para no quedarse sin memoria
# durante el entrenamiento de RandomForest.
spark = (
    SparkSession.builder
    .appName("u1-producto-consolidado-llsw3")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

RUTA_DATOS = "/opt/data/TRCU.csv"

esquema_flujos = StructType([
    StructField("flow_id", StringType(), True),
    StructField("src_addr", StringType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_addr", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("ip_prot", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("flow_duration", DoubleType(), True),
    StructField("down_up_ratio", DoubleType(), True),
    StructField("pkt_len_max", DoubleType(), True),
    StructField("pkt_len_min", DoubleType(), True),
    StructField("pkt_len_mean", DoubleType(), True),
    StructField("pkt_len_var", DoubleType(), True),
    StructField("pkt_len_std", DoubleType(), True),
    StructField("bytes_per_s", DoubleType(), True),
    StructField("pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_per_s", DoubleType(), True),
    StructField("bwd_pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_cnt", IntegerType(), True),
    StructField("fwd_pkt_len_tot", DoubleType(), True),
    StructField("fwd_pkt_len_max", DoubleType(), True),
    StructField("fwd_pkt_len_min", DoubleType(), True),
    StructField("fwd_pkt_len_mean", DoubleType(), True),
    StructField("fwd_pkt_len_std", DoubleType(), True),
    StructField("fwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("fwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("fwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_len_tot", DoubleType(), True),
    StructField("bwd_pkt_len_max", DoubleType(), True),
    StructField("bwd_pkt_len_min", DoubleType(), True),
    StructField("bwd_pkt_len_mean", DoubleType(), True),
    StructField("bwd_pkt_len_std", DoubleType(), True),
    StructField("bwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("bwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("bwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("iat_max", DoubleType(), True),
    StructField("iat_min", DoubleType(), True),
    StructField("iat_mean", DoubleType(), True),
    StructField("iat_std", DoubleType(), True),
    StructField("fwd_iat_tot", DoubleType(), True),
    StructField("fwd_iat_max", DoubleType(), True),
    StructField("fwd_iat_min", DoubleType(), True),
    StructField("fwd_iat_mean", DoubleType(), True),
    StructField("fwd_iat_std", DoubleType(), True),
    StructField("bwd_iat_tot", DoubleType(), True),
    StructField("bwd_iat_max", DoubleType(), True),
    StructField("bwd_iat_min", DoubleType(), True),
    StructField("bwd_iat_mean", DoubleType(), True),
    StructField("bwd_iat_std", DoubleType(), True),
    StructField("active_max", DoubleType(), True),
    StructField("active_min", DoubleType(), True),
    StructField("active_mean", DoubleType(), True),
    StructField("active_std", DoubleType(), True),
    StructField("idle_max", DoubleType(), True),
    StructField("idle_min", DoubleType(), True),
    StructField("idle_mean", DoubleType(), True),
    StructField("idle_std", DoubleType(), True),
    StructField("flag_SYN", IntegerType(), True),
    StructField("flag_fin", IntegerType(), True),
    StructField("flag_rst", IntegerType(), True),
    StructField("flag_ack", IntegerType(), True),
    StructField("flag_psh", IntegerType(), True),
    StructField("fwd_flag_psh", IntegerType(), True),
    StructField("bwd_flag_psh", IntegerType(), True),
    StructField("flag_urg", IntegerType(), True),
    StructField("fwd_flag_urg", IntegerType(), True),
    StructField("bwd_flag_urg", IntegerType(), True),
    StructField("flag_cwr", IntegerType(), True),
    StructField("flag_ece", IntegerType(), True),
    StructField("fwd_bulk_bytes_mean", DoubleType(), True),
    StructField("fwd_bulk_pkt_mean", DoubleType(), True),
    StructField("fwd_bulk_rate_mean", DoubleType(), True),
    StructField("bwd_bulk_bytes_mean", DoubleType(), True),
    StructField("bwd_bulk_pkt_mean", DoubleType(), True),
    StructField("bwd_bulk_rate_mean", DoubleType(), True),
    StructField("fwd_subflow_bytes_mean", DoubleType(), True),
    StructField("fwd_subflow_pkt_mean", DoubleType(), True),
    StructField("bwd_subflow_bytes_mean", DoubleType(), True),
    StructField("bwd_subflow_pkt_mean", DoubleType(), True),
    StructField("fwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("bwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("label", StringType(), True),
])

df = spark.read.csv(RUTA_DATOS, header=True, schema=esquema_flujos).cache()

with open(RUTA_DATOS, "r", encoding="utf-8") as f:
    cabecera_real = f.readline().strip().split(",")
assert cabecera_real == [c.name for c in esquema_flujos.fields], (
    "El orden del esquema no coincide con el header real del CSV — revisar antes de continuar."
)

df.printSchema()
df.show(5, truncate=False)
print("Filas totales:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 02:54:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/11 02:54:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


root
 |-- flow_id: string (nullable = true)
 |-- src_addr: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_addr: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- ip_prot: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- pkt_len_max: double (nullable = true)
 |-- pkt_len_min: double (nullable = true)
 |-- pkt_len_mean: double (nullable = true)
 |-- pkt_len_var: double (nullable = true)
 |-- pkt_len_std: double (nullable = true)
 |-- bytes_per_s: double (nullable = true)
 |-- pkt_per_s: double (nullable = true)
 |-- fwd_pkt_per_s: double (nullable = true)
 |-- bwd_pkt_per_s: double (nullable = true)
 |-- fwd_pkt_cnt: integer (nullable = true)
 |-- fwd_pkt_len_tot: double (nullable = true)
 |-- fwd_pkt_len_max: double (nullable = true)
 |-- fwd_pkt_len_min: double (nullable = true)
 |-- fwd_pkt_len_mean: double (nullable = true)
 |

+-----------------------------------------+--------------+--------+--------------+--------+-------+----------------+-------------+-------------+-----------+-----------+------------+-------------+-----------+------------+---------+-------------+-------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+--------+-------+------------+-------------+-----------+-----------+-----------+------------+-------------+-----------+-----------+-----------+------------+-------------+----------+----------+-----------+----------+--------+--------+---------+--------+--------+--------+--------+--------+--------+------------+------------+--------+------------+------------+--------+--------+-------------------+-----------------+------------------

Filas totales: 397354


### Exploración inicial (EDA) general

Estadísticos y distribuciones que las cuatro dimensiones comparten como punto de partida: nulos en columnas clave, distribución por protocolo, y las variables objetivo de cada dimensión.

In [2]:
print("Nulos por columna clave:")
for columna in ["bytes_per_s", "flow_duration", "dst_port", "down_up_ratio", "pkt_len_mean", "iat_mean", "ip_prot"]:
    n_nulos = df.filter(F.col(columna).isNull()).count()
    print(f"  {columna}: {n_nulos} nulos")

print("\nDistribucion por protocolo (ip_prot):")
df.groupBy("ip_prot").count().orderBy(F.desc("count")).show()

print("Top 15 puertos de destino mas frecuentes:")
df.groupBy("dst_port").count().orderBy(F.desc("count")).show(15)

Nulos por columna clave:


  bytes_per_s: 0 nulos
  flow_duration: 0 nulos


  dst_port: 0 nulos
  down_up_ratio: 0 nulos


  pkt_len_mean: 0 nulos
  iat_mean: 0 nulos


  ip_prot: 0 nulos

Distribucion por protocolo (ip_prot):


+-------+------+
|ip_prot| count|
+-------+------+
|     17|374483|
|      6| 22579|
|      2|   243|
|      1|    49|
+-------+------+

Top 15 puertos de destino mas frecuentes:


+--------+------+
|dst_port| count|
+--------+------+
|   10001|157470|
|   10002|102610|
|    5355| 32630|
|    5353| 25969|
|    8014|  8621|
|   50160|  4862|
|     137|  4572|
|     138|  3692|
|   56700|  3173|
|     443|  2849|
|   18070|  1886|
|      67|  1875|
|    3289|  1579|
|   57621|  1121|
|   21741|  1088|
+--------+------+
only showing top 15 rows


In [3]:
print("bytes_per_s (Nick) -> min/p25/mediana/p75/max:", df.approxQuantile("bytes_per_s", [0.0, 0.25, 0.5, 0.75, 1.0], 0.01))
print("flow_duration (Jhan) -> min/p25/mediana/p75/max:", df.approxQuantile("flow_duration", [0.0, 0.25, 0.5, 0.75, 1.0], 0.01))
print("down_up_ratio (David) -> min/p33/mediana/p66/max:", df.approxQuantile("down_up_ratio", [0.0, 0.33, 0.5, 0.66, 1.0], 0.01))

bytes_per_s (Nick) -> min/p25/mediana/p75/max: [0.0, 0.0, 0.0, 0.0, 2760000000.0]


flow_duration (Jhan) -> min/p25/mediana/p75/max: [0.0, 0.0, 0.0, 1.0, 119999999.0]


down_up_ratio (David) -> min/p33/mediana/p66/max: [0.0, 0.0, 0.0, 0.0, 2.4]


---
## Dimensión 1 (Nick) — Volumen de tráfico por flujo (`bytes_per_s`), regresión

### Fase 3 — Preparación de los datos

In [4]:
df_nick = (
    df
    .withColumn(
        "protocolo",
        F.when(F.col("ip_prot") == 6, F.lit("TCP"))
         .when(F.col("ip_prot") == 17, F.lit("UDP"))
         .otherwise(F.lit("OTRO"))
    )
    .filter(F.col("bytes_per_s").isNotNull())
)

linea_base_ingenua = df.select(F.avg("bytes_per_s")).first()[0]
print("Linea base ingenua (promedio historico de bytes_per_s):", linea_base_ingenua)

df_nick.explain(True)  # confirmar en que punto Spark deja de ser perezoso

resumen_volumen = (
    df_nick.groupBy("protocolo")
    .agg(
        F.avg("bytes_per_s").alias("bytes_per_s_prom"),
        F.max("bytes_per_s").alias("bytes_per_s_max"),
        F.count("*").alias("n_flujos"),
    )
)
resumen_volumen.show()

Linea base ingenua (promedio historico de bytes_per_s): 550136.9126835192
== Parsed Logical Plan ==
'Filter 'isNotNull('bytes_per_s)
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
   +- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fields] csv

== Analyz

+---------+------------------+---------------+--------+
|protocolo|  bytes_per_s_prom|bytes_per_s_max|n_flujos|
+---------+------------------+---------------+--------+
|     OTRO|  8538.14210946233| 1290322.580645|     292|
|      UDP|209179.59557949615|          2.4E9|  374483|
|      TCP| 6212073.483039771|         2.76E9|   22579|
+---------+------------------+---------------+--------+



In [5]:
df_dedup = df_nick.dropDuplicates(["flow_id"])

df_limpio_nick = df_dedup.na.fill({
    "bytes_per_s": 0.0,
    "pkt_len_mean": 0.0,
    "iat_mean": 0.0,
    "active_mean": 0.0,
    "idle_mean": 0.0,
})

RUTA_SALIDA_NICK = "/opt/artifacts/nick/flujos_particionado"
(
    df_limpio_nick.write.mode("overwrite")
    .partitionBy("protocolo")
    .parquet(RUTA_SALIDA_NICK)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA_NICK)
print("Filas tras deduplicacion:", df_limpio_nick.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("protocolo") == "TCP").explain(True)  # confirmar PartitionFilters

Filas tras deduplicacion: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('protocolo, TCP)
+- Relation [flow_id#38928,src_addr#38929,src_port#38930,dst_addr#38931,dst_port#38932,ip_prot#38933,timestamp#38934L,flow_duration#38935,down_up_ratio#38936,pkt_len_max#38937,pkt_len_min#38938,pkt_len_mean#38939,pkt_len_var#38940,pkt_len_std#38941,bytes_per_s#38942,pkt_per_s#38943,fwd_pkt_per_s#38944,bwd_pkt_per_s#38945,fwd_pkt_cnt#38946,fwd_pkt_len_tot#38947,fwd_pkt_len_max#38948,fwd_pkt_len_min#38949,fwd_pkt_len_mean#38950,fwd_pkt_len_std#38951,fwd_pkt_hdr_len_tot#38952,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_per_s: double, fwd_pkt_cn

In [6]:
from pyspark.ml.feature import VectorAssembler

predictores_nick = [
    "ip_prot", "flow_duration", "pkt_len_mean", "pkt_len_std",
    "fwd_pkt_len_mean", "bwd_pkt_len_mean", "iat_mean",
    "active_mean", "idle_mean", "flag_SYN", "flag_ack",
    "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

ensamblador = VectorAssembler(inputCols=predictores_nick, outputCol="features", handleInvalid="skip")
dataset_ml = (
    ensamblador.transform(df_limpio_nick)
    .select("features", F.col("bytes_per_s").alias("valor_real"))
)

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())

Filas de entrenamiento: 106874  / Filas de prueba: 26437


### Fase 4 — Modelado

In [7]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

configuraciones_nick = {
    "LinearRegression base": LinearRegression(featuresCol="features", labelCol="valor_real"),
    "LinearRegression + Ridge (L2)": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.0),
    "LinearRegression + Lasso (L1)": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=1.0),
    "LinearRegression + Elastic Net": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.5),
    "RandomForestRegressor": RandomForestRegressor(featuresCol="features", labelCol="valor_real", seed=42),
}

modelos_entrenados_nick = {}
predicciones_nick = {}
for nombre, estimador in configuraciones_nick.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_nick[nombre] = modelo
    predicciones_nick[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)

26/09/11 02:55:09 WARN Instrumentation: [fae1fcf9] regParam is zero, which might cause numerical instability and overfitting.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Entrenado: LinearRegression base


Entrenado: LinearRegression + Ridge (L2)


Entrenado: LinearRegression + Lasso (L1)


Entrenado: LinearRegression + Elastic Net


Entrenado: RandomForestRegressor


### Fase 5 — Evaluación

In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

ev_rmse = RegressionEvaluator(labelCol="valor_real", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="valor_real", metricName="r2")
ev_mae = RegressionEvaluator(labelCol="valor_real", metricName="mae")

pred_baseline = df_test.withColumn("prediction", F.lit(linea_base_ingenua))
rmse_base = ev_rmse.evaluate(pred_baseline)
print(f"Linea base ingenua -> RMSE={rmse_base:.4f}\n")

resultados_nick = {}
for nombre, pred in predicciones_nick.items():
    rmse = ev_rmse.evaluate(pred)
    r2 = ev_r2.evaluate(pred)
    mae = ev_mae.evaluate(pred)
    supera_base = "SI" if rmse < rmse_base else "NO"
    resultados_nick[nombre] = rmse
    print(f"{nombre:32s} RMSE={rmse:.4f}  R2={r2:.4f}  MAE={mae:.4f}  (supera linea base: {supera_base})")

nombre_ganador_nick = min(resultados_nick, key=resultados_nick.get)
print(f"\nModelo ganador (menor RMSE): {nombre_ganador_nick}")

modelo_ganador_nick = modelos_entrenados_nick[nombre_ganador_nick]
modelo_ganador_nick.write().overwrite().save("/opt/artifacts/nick/modelo_volumen")
print("Modelo ganador guardado en /opt/artifacts/nick/modelo_volumen")

Linea base ingenua -> RMSE=34245209.1615



LinearRegression base            RMSE=33787402.3774  R2=0.0259  MAE=3159715.7261  (supera linea base: SI)


LinearRegression + Ridge (L2)    RMSE=33787402.3733  R2=0.0259  MAE=3159715.7349  (supera linea base: SI)


LinearRegression + Lasso (L1)    RMSE=33786414.8927  R2=0.0260  MAE=3164290.7875  (supera linea base: SI)


LinearRegression + Elastic Net   RMSE=33786414.8964  R2=0.0260  MAE=3164290.7783  (supera linea base: SI)


RandomForestRegressor            RMSE=16116512.4038  R2=0.7784  MAE=638386.1837  (supera linea base: SI)

Modelo ganador (menor RMSE): RandomForestRegressor


Modelo ganador guardado en /opt/artifacts/nick/modelo_volumen


**Hallazgos (Nick):** la deduplicación por `flow_id` eliminó el 66.5% de los registros (397 354 → 133 311) — casi dos tercios de las capturas de Suricata llegan repetidas al pipeline batch. `bytes_per_s` tiene una relación fuertemente no lineal con las features del flujo: los 4 modelos lineales apenas superan la línea base (R² ≈ 0.026), mientras que `RandomForestRegressor` alcanza **R² = 0.7784** y reduce el RMSE en ~53% (16.1M vs. 34.2M), cumpliendo el criterio de éxito de la Fase 1 (R² > 0.5). UDP domina en número de flujos (374 483 de 397 354, ~94%), pero TCP tiene el `bytes_per_s` promedio más alto (6.21M vs. 209K de UDP).

---
## Dimensión 2 (Jhan) — Duración del flujo de red (`flow_duration`), regresión

### Fase 3 — Preparación de los datos

In [9]:
percentiles_duracion = df.approxQuantile("flow_duration", [0.0, 0.25, 0.5, 0.75, 1.0], 0.01)
linea_base_ingenua = percentiles_duracion[2]  # mediana historica completa
print("Linea base ingenua (mediana historica de flow_duration):", linea_base_ingenua)

df_jhan = (
    df
    .withColumn(
        "protocolo",
        F.when(F.col("ip_prot") == 6, F.lit("TCP"))
         .when(F.col("ip_prot") == 17, F.lit("UDP"))
         .otherwise(F.lit("OTRO"))
    )
    .filter(F.col("flow_duration").isNotNull() & (F.col("flow_duration") > 0))
)

df_jhan.explain(True)  # confirmar en que punto Spark deja de ser perezoso

resumen_duracion = (
    df_jhan.groupBy("protocolo")
    .agg(
        F.avg("flow_duration").alias("duracion_prom_us"),
        F.expr("percentile_approx(flow_duration, 0.5)").alias("duracion_mediana_us"),
        F.count("*").alias("n_flujos"),
    )
)
resumen_duracion.show()

Linea base ingenua (mediana historica de flow_duration): 0.0
== Parsed Logical Plan ==
'Filter 'and('isNotNull('flow_duration), '`>`('flow_duration, 0))
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
   +- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fi

+---------+--------------------+-------------------+--------+
|protocolo|    duracion_prom_us|duracion_mediana_us|n_flujos|
+---------+--------------------+-------------------+--------+
|     OTRO|1.2868307241666667E7|          3501615.0|     240|
|      UDP|4.3848324977936536E7|          9211691.0|   87203|
|      TCP|2.5283146336503245E7|           228220.0|   15563|
+---------+--------------------+-------------------+--------+



In [10]:
df_dedup = df_jhan.dropDuplicates(["flow_id"])

df_limpio_jhan = df_dedup.na.fill({
    "flow_duration": 0.0,
    "iat_mean": 0.0,
    "fwd_iat_mean": 0.0,
    "bwd_iat_mean": 0.0,
    "fwd_pkt_cnt": 0,
    "bwd_pkt_cnt": 0,
})

RUTA_SALIDA_JHAN = "/opt/artifacts/jhan/flujos_particionado"
(
    df_limpio_jhan.write.mode("overwrite")
    .partitionBy("protocolo")
    .parquet(RUTA_SALIDA_JHAN)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA_JHAN)
print("Filas tras deduplicacion:", df_limpio_jhan.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("protocolo") == "TCP").explain(True)  # confirmar PartitionFilters

Filas tras deduplicacion: 39728


Filas leidas de vuelta desde Parquet: 39728
== Parsed Logical Plan ==
'Filter '`=`('protocolo, TCP)
+- Relation [flow_id#124225,src_addr#124226,src_port#124227,dst_addr#124228,dst_port#124229,ip_prot#124230,timestamp#124231L,flow_duration#124232,down_up_ratio#124233,pkt_len_max#124234,pkt_len_min#124235,pkt_len_mean#124236,pkt_len_var#124237,pkt_len_std#124238,bytes_per_s#124239,pkt_per_s#124240,fwd_pkt_per_s#124241,bwd_pkt_per_s#124242,fwd_pkt_cnt#124243,fwd_pkt_len_tot#124244,fwd_pkt_len_max#124245,fwd_pkt_len_min#124246,fwd_pkt_len_mean#124247,fwd_pkt_len_std#124248,fwd_pkt_hdr_len_tot#124249,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_p

In [11]:
predictores_jhan = [
    "ip_prot", "iat_mean", "iat_std", "fwd_iat_mean", "bwd_iat_mean",
    "fwd_pkt_cnt", "bwd_pkt_cnt", "pkt_len_mean", "flag_SYN", "flag_ack",
    "active_mean", "idle_mean", "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

ensamblador = VectorAssembler(inputCols=predictores_jhan, outputCol="features", handleInvalid="skip")
dataset_ml = (
    ensamblador.transform(df_limpio_jhan)
    .select("features", F.col("flow_duration").alias("valor_real"))
)

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())

Filas de entrenamiento: 31974  / Filas de prueba: 7754


### Fase 4 — Modelado

In [12]:
configuraciones_jhan = {
    "LinearRegression base": LinearRegression(featuresCol="features", labelCol="valor_real"),
    "LinearRegression + regularizacion": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.5),
    "RandomForestRegressor": RandomForestRegressor(featuresCol="features", labelCol="valor_real", seed=42),
}

modelos_entrenados_jhan = {}
predicciones_jhan = {}
for nombre, estimador in configuraciones_jhan.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_jhan[nombre] = modelo
    predicciones_jhan[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)

26/09/11 02:55:30 WARN Instrumentation: [777ff398] regParam is zero, which might cause numerical instability and overfitting.


Entrenado: LinearRegression base


Entrenado: LinearRegression + regularizacion


Entrenado: RandomForestRegressor


### Fase 5 — Evaluación

In [13]:
ev_rmse = RegressionEvaluator(labelCol="valor_real", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="valor_real", metricName="r2")
ev_mae = RegressionEvaluator(labelCol="valor_real", metricName="mae")

pred_baseline = df_test.withColumn("prediction", F.lit(linea_base_ingenua))
rmse_base = ev_rmse.evaluate(pred_baseline)
print(f"Linea base ingenua (mediana) -> RMSE={rmse_base:.4f}\n")

resultados_jhan = {}
for nombre, pred in predicciones_jhan.items():
    rmse = ev_rmse.evaluate(pred)
    r2 = ev_r2.evaluate(pred)
    mae = ev_mae.evaluate(pred)
    reduccion_pct = (1 - rmse / rmse_base) * 100 if rmse_base else float("nan")
    resultados_jhan[nombre] = rmse
    print(f"{nombre:32s} RMSE={rmse:.4f}  R2={r2:.4f}  MAE={mae:.4f}  (reduccion vs base: {reduccion_pct:.1f}%)")

nombre_ganador_jhan = min(resultados_jhan, key=resultados_jhan.get)
print(f"\nModelo ganador (menor RMSE): {nombre_ganador_jhan}")

modelo_ganador_jhan = modelos_entrenados_jhan[nombre_ganador_jhan]
modelo_ganador_jhan.write().overwrite().save("/opt/artifacts/jhan/modelo_duracion")
print("Modelo ganador guardado en /opt/artifacts/jhan/modelo_duracion")

Linea base ingenua (mediana) -> RMSE=24653396.5808



LinearRegression base            RMSE=23136259.7375  R2=0.0054  MAE=4342154.1704  (reduccion vs base: 6.2%)


LinearRegression + regularizacion RMSE=23280744.8503  R2=-0.0070  MAE=4355295.4328  (reduccion vs base: 5.6%)


RandomForestRegressor            RMSE=7387473.0216  R2=0.8986  MAE=2332560.4385  (reduccion vs base: 70.0%)

Modelo ganador (menor RMSE): RandomForestRegressor


Modelo ganador guardado en /opt/artifacts/jhan/modelo_duracion


**Hallazgos (Jhan):** tras filtrar `flow_duration > 0` y deduplicar, solo quedan 39 728 de 397 354 flujos (~10%) — la mediana histórica completa es 0, es decir, más de la mitad de los flujos capturados son instantáneos. `RandomForestRegressor` reduce el RMSE en **70.0%** frente a la línea base (7.39M vs. 24.65M) con **R² = 0.8986**, superando ampliamente el criterio de éxito (≥20%); los modelos lineales apenas se mueven (R² ≈ 0.005). Por protocolo: TCP tiene duración mediana mucho menor (228 220 µs ≈ 0.23s, n=15 563) que UDP (9 211 691 µs ≈ 9.2s, n=87 203) — señal directa para dimensionar ventanas de sesión por protocolo en Kafka (U2).

---
## Dimensión 3 (Henyelrey) — Tipo de servicio vs. catálogo IANA, clasificación

### Fase 3 — Preparación de los datos

In [14]:
catalogo_iana = spark.createDataFrame([
    (20, "ftp"), (21, "ftp"), (22, "ssh"), (23, "telnet"), (25, "correo"),
    (53, "dns"), (67, "dhcp"), (68, "dhcp"), (80, "web"), (110, "correo"),
    (123, "ntp"), (143, "correo"), (161, "snmp"), (194, "chat"), (443, "web"),
    (445, "archivos"), (465, "correo"), (587, "correo"), (993, "correo"),
    (995, "correo"), (1433, "bd"), (1521, "bd"), (1723, "vpn"), (3306, "bd"),
    (3389, "escritorio_remoto"), (5432, "bd"), (5900, "escritorio_remoto"),
    (8080, "web"), (8443, "web"),
], ["puerto", "categoria_servicio_iana"])
# Catalogo simplificado de puertos "well-known" (IANA) — servicios mas comunes de campus.
# Registro oficial completo: iana.org/assignments/service-names-port-numbers

df_henyelrey = (
    df.join(catalogo_iana, df.dst_port == catalogo_iana.puerto, "left")
    .withColumn(
        "categoria_servicio_ref",
        F.coalesce(F.col("categoria_servicio_iana"), F.lit("otro_desconocido"))
    )
    .drop("puerto", "categoria_servicio_iana")
)

df_henyelrey.explain(True)

distribucion_servicio = df_henyelrey.groupBy("categoria_servicio_ref").agg(F.count("*").alias("n_flujos"))
distribucion_servicio.orderBy(F.desc("n_flujos")).show(30, truncate=False)

== Parsed Logical Plan ==
Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 61 more fields]
   +- Join LeftOuter, (cast(dst_port#4 as bigint) = puerto#172540L)
      :- Relation [flow_id#0,src_ad

+----------------------+--------+
|categoria_servicio_ref|n_flujos|
+----------------------+--------+
|otro_desconocido      |390530  |
|web                   |3046    |
|dhcp                  |2745    |
|dns                   |963     |
|snmp                  |43      |
|ntp                   |21      |
|archivos              |6       |
+----------------------+--------+



In [15]:
df_dedup = df_henyelrey.dropDuplicates(["flow_id"])
df_limpio_henyelrey = df_dedup.na.drop(subset=["categoria_servicio_ref"])

RUTA_SALIDA_HENYELREY = "/opt/artifacts/henyelrey/flujos_particionado"
(
    df_limpio_henyelrey.write.mode("overwrite")
    .partitionBy("categoria_servicio_ref")
    .parquet(RUTA_SALIDA_HENYELREY)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA_HENYELREY)
print("Filas tras limpieza:", df_limpio_henyelrey.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("categoria_servicio_ref") == "web").explain(True)  # PartitionFilters

Filas tras limpieza: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('categoria_servicio_ref, web)
+- Relation [flow_id#178706,src_addr#178707,src_port#178708,dst_addr#178709,dst_port#178710,ip_prot#178711,timestamp#178712L,flow_duration#178713,down_up_ratio#178714,pkt_len_max#178715,pkt_len_min#178716,pkt_len_mean#178717,pkt_len_var#178718,pkt_len_std#178719,bytes_per_s#178720,pkt_per_s#178721,fwd_pkt_per_s#178722,bwd_pkt_per_s#178723,fwd_pkt_cnt#178724,fwd_pkt_len_tot#178725,fwd_pkt_len_max#178726,fwd_pkt_len_min#178727,fwd_pkt_len_mean#178728,fwd_pkt_len_std#178729,fwd_pkt_hdr_len_tot#178730,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: dou

In [16]:
from pyspark.ml.feature import StringIndexer

# Importante: dst_port/src_port quedan fuera de los predictores a proposito — el objetivo
# es clasificar el servicio a partir del COMPORTAMIENTO del flujo, no leyendo el puerto
# (que es, de hecho, la fuente de la propia etiqueta de referencia).
predictores_henyelrey = [
    "ip_prot", "flow_duration", "pkt_len_mean", "pkt_len_std", "bytes_per_s",
    "fwd_pkt_len_mean", "bwd_pkt_len_mean", "iat_mean", "flag_SYN", "flag_ack",
    "flag_psh", "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

indexador = StringIndexer(inputCol="categoria_servicio_ref", outputCol="label_idx")
modelo_indexador = indexador.fit(df_limpio_henyelrey)
df_indexado = modelo_indexador.transform(df_limpio_henyelrey)

ensamblador = VectorAssembler(inputCols=predictores_henyelrey, outputCol="features", handleInvalid="skip")
dataset_ml = ensamblador.transform(df_indexado).select("features", "label_idx")

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())

Filas de entrenamiento: 106786  / Filas de prueba: 26525


### Fase 4 — Modelado

In [17]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

configuraciones_henyelrey = {
    "LogisticRegression base": LogisticRegression(featuresCol="features", labelCol="label_idx"),
    "LogisticRegression + regularizacion": LogisticRegression(featuresCol="features", labelCol="label_idx", regParam=0.1, elasticNetParam=0.5),
    "RandomForestClassifier": RandomForestClassifier(featuresCol="features", labelCol="label_idx", seed=42),
}

modelos_entrenados_henyelrey = {}
predicciones_henyelrey = {}
for nombre, estimador in configuraciones_henyelrey.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_henyelrey[nombre] = modelo
    predicciones_henyelrey[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)

Entrenado: LogisticRegression base


Entrenado: LogisticRegression + regularizacion


Entrenado: RandomForestClassifier


### Fase 5 — Evaluación

In [18]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ev_acc = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="f1")
ev_prec = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="weightedPrecision")

clase_mayoritaria = df_test.groupBy("label_idx").count().orderBy(F.desc("count")).first()
frac_mayoritaria = clase_mayoritaria["count"] / df_test.count()
print(f"Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy={frac_mayoritaria:.4f}\n")

resultados_henyelrey = {}
for nombre, pred in predicciones_henyelrey.items():
    acc = ev_acc.evaluate(pred)
    f1 = ev_f1.evaluate(pred)
    prec = ev_prec.evaluate(pred)
    diferencia_pp = (acc - frac_mayoritaria) * 100
    resultados_henyelrey[nombre] = f1
    print(f"{nombre:38s} Accuracy={acc:.4f}  F1={f1:.4f}  Precision={prec:.4f}  (+{diferencia_pp:.1f} pp vs linea base)")

nombre_ganador_henyelrey = max(resultados_henyelrey, key=resultados_henyelrey.get)
print(f"\nModelo ganador (mayor F1 ponderado): {nombre_ganador_henyelrey}")

mejor_pred_henyelrey = predicciones_henyelrey[nombre_ganador_henyelrey]
mejor_pred_henyelrey.groupBy("label_idx", "prediction").count().orderBy("label_idx", "prediction").show(50)

modelo_ganador_henyelrey = modelos_entrenados_henyelrey[nombre_ganador_henyelrey]
modelo_ganador_henyelrey.write().overwrite().save("/opt/artifacts/henyelrey/modelo_servicio")
print("Modelo ganador guardado en /opt/artifacts/henyelrey/modelo_servicio")

Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy=0.9696



LogisticRegression base                Accuracy=0.9873  F1=0.9865  Precision=0.9860  (+1.8 pp vs linea base)


LogisticRegression + regularizacion    Accuracy=0.9686  F1=0.9551  Precision=0.9457  (+-0.1 pp vs linea base)


RandomForestClassifier                 Accuracy=0.9931  F1=0.9927  Precision=0.9924  (+2.4 pp vs linea base)

Modelo ganador (mayor F1 ponderado): RandomForestClassifier


+---------+----------+-----+
|label_idx|prediction|count|
+---------+----------+-----+
|      0.0|       0.0|25692|
|      0.0|       1.0|   23|
|      0.0|       2.0|    4|
|      1.0|       0.0|   96|
|      1.0|       1.0|  485|
|      1.0|       2.0|    3|
|      2.0|       0.0|    1|
|      2.0|       1.0|   41|
|      2.0|       2.0|  166|
|      3.0|       0.0|    5|
|      4.0|       0.0|    4|
|      5.0|       0.0|    5|
+---------+----------+-----+



Modelo ganador guardado en /opt/artifacts/henyelrey/modelo_servicio


**Hallazgos (Henyelrey):** el catálogo IANA simplificado (29 puertos) solo etiqueta al **1.7%** de los flujos en una categoría conocida (6 824 de 397 354: web=3 046, dhcp=2 745, dns=963, snmp=43, ntp=21, archivos=6) — el **98.3%** cae en `otro_desconocido`, porque los puertos reales más frecuentes del campus (10001, 10002, 5355, 5353, 8014...) no están en el catálogo. Por eso la línea base ya alcanza accuracy=0.9696, y aunque `RandomForestClassifier` gana (Accuracy=0.9931, F1=0.9927), la mejora es solo **+2.4 pp** — **no cumple** el criterio de éxito de la Fase 1 (objetivo +15-20 pp). La matriz de confusión sí distingue bien las clases con volumen suficiente (web: ~83% correctas, dhcp: ~80% correctas). **Recomendación:** ampliar el catálogo IANA antes de calibrar la dimensión U2.

---
## Dimensión 4 (David) — Dirección dominante del flujo (`down_up_ratio`), clasificación

### Fase 3 — Preparación de los datos

In [19]:
cuantiles = df.approxQuantile("down_up_ratio", [0.33, 0.66], 0.01)
p33, p66 = cuantiles[0], cuantiles[1]
print("Cortes calculados sobre el historico -> p33:", p33, " p66:", p66)

df_david = df.withColumn(
    "categoria_direccion",
    F.when(F.col("down_up_ratio") < p33, F.lit("carga_dominante"))
     .when(F.col("down_up_ratio") > p66, F.lit("descarga_dominante"))
     .otherwise(F.lit("balanceado"))
)

df_david.explain(True)

distribucion_direccion = df_david.groupBy("categoria_direccion").agg(F.count("*").alias("n_flujos"))
distribucion_direccion.show()

Cortes calculados sobre el historico -> p33: 0.0  p66: 0.0


== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(categoria_direccion, CASE WHEN '`<`('down_up_ratio, 0.0) THEN carga_dominante WHEN '`>`('down_up_ratio, 0.0) THEN descarga_dominante ELSE balanceado END, None)]
+- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fields] csv

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, 

In [20]:
df_dedup = df_david.dropDuplicates(["flow_id"])
df_limpio_david = df_dedup.na.drop(subset=["down_up_ratio", "categoria_direccion"])

RUTA_SALIDA_DAVID = "/opt/artifacts/david/flujos_particionado"
(
    df_limpio_david.write.mode("overwrite")
    .partitionBy("categoria_direccion")
    .parquet(RUTA_SALIDA_DAVID)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA_DAVID)
print("Filas tras limpieza:", df_limpio_david.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("categoria_direccion") == "descarga_dominante").explain(True)

Filas tras limpieza: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('categoria_direccion, descarga_dominante)
+- Relation [flow_id#248880,src_addr#248881,src_port#248882,dst_addr#248883,dst_port#248884,ip_prot#248885,timestamp#248886L,flow_duration#248887,down_up_ratio#248888,pkt_len_max#248889,pkt_len_min#248890,pkt_len_mean#248891,pkt_len_var#248892,pkt_len_std#248893,bytes_per_s#248894,pkt_per_s#248895,fwd_pkt_per_s#248896,bwd_pkt_per_s#248897,fwd_pkt_cnt#248898,fwd_pkt_len_tot#248899,fwd_pkt_len_max#248900,fwd_pkt_len_min#248901,fwd_pkt_len_mean#248902,fwd_pkt_len_std#248903,fwd_pkt_hdr_len_tot#248904,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pk

In [21]:
# Se excluyen a proposito las columnas de tamano/bytes (pkt_len_*, *_pkt_len_tot,
# bytes_per_s, *_bulk_*) porque down_up_ratio se deriva directamente de ellas — usarlas
# como predictor seria fuga de informacion. Los predictores se limitan a protocolo,
# tiempos y banderas: comportamiento, no volumen.
predictores_david = [
    "ip_prot", "flow_duration", "iat_mean", "iat_std", "fwd_iat_mean", "bwd_iat_mean",
    "flag_SYN", "flag_ack", "flag_psh", "active_mean", "idle_mean",
    "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

indexador = StringIndexer(inputCol="categoria_direccion", outputCol="label_idx")
modelo_indexador = indexador.fit(df_limpio_david)
df_indexado = modelo_indexador.transform(df_limpio_david)

ensamblador = VectorAssembler(inputCols=predictores_david, outputCol="features", handleInvalid="skip")
dataset_ml = ensamblador.transform(df_indexado).select("features", "label_idx")

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())

Filas de entrenamiento: 106817  / Filas de prueba: 26494


### Fase 4 — Modelado

In [22]:
configuraciones_david = {
    "LogisticRegression base": LogisticRegression(featuresCol="features", labelCol="label_idx"),
    "LogisticRegression + regularizacion": LogisticRegression(featuresCol="features", labelCol="label_idx", regParam=0.1, elasticNetParam=0.5),
    "RandomForestClassifier": RandomForestClassifier(featuresCol="features", labelCol="label_idx", seed=42),
}

modelos_entrenados_david = {}
predicciones_david = {}
for nombre, estimador in configuraciones_david.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_david[nombre] = modelo
    predicciones_david[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)

Entrenado: LogisticRegression base


Entrenado: LogisticRegression + regularizacion


Entrenado: RandomForestClassifier


### Fase 5 — Evaluación

In [23]:
ev_acc = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="f1")
ev_prec = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="weightedPrecision")

clase_mayoritaria = df_test.groupBy("label_idx").count().orderBy(F.desc("count")).first()
frac_mayoritaria = clase_mayoritaria["count"] / df_test.count()
print(f"Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy={frac_mayoritaria:.4f}\n")

resultados_david = {}
for nombre, pred in predicciones_david.items():
    acc = ev_acc.evaluate(pred)
    f1 = ev_f1.evaluate(pred)
    prec = ev_prec.evaluate(pred)
    resultados_david[nombre] = f1
    print(f"{nombre:38s} Accuracy={acc:.4f}  F1={f1:.4f}  Precision={prec:.4f}")

nombre_ganador_david = max(resultados_david, key=resultados_david.get)
print(f"\nModelo ganador (mayor F1 ponderado): {nombre_ganador_david}")

mejor_pred_david = predicciones_david[nombre_ganador_david]
mejor_pred_david.groupBy("label_idx", "prediction").count().orderBy("label_idx", "prediction").show(50)

modelo_ganador_david = modelos_entrenados_david[nombre_ganador_david]
modelo_ganador_david.write().overwrite().save("/opt/artifacts/david/modelo_direccion")
print("Modelo ganador guardado en /opt/artifacts/david/modelo_direccion")

Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy=0.9958



LogisticRegression base                Accuracy=0.9977  F1=0.9973  Precision=0.9977


LogisticRegression + regularizacion    Accuracy=0.9958  F1=0.9938  Precision=0.9917


RandomForestClassifier                 Accuracy=0.9978  F1=0.9974  Precision=0.9978

Modelo ganador (mayor F1 ponderado): RandomForestClassifier


+---------+----------+-----+
|label_idx|prediction|count|
+---------+----------+-----+
|      0.0|       0.0|26384|
|      1.0|       0.0|   59|
|      1.0|       1.0|   51|
+---------+----------+-----+



Modelo ganador guardado en /opt/artifacts/david/modelo_direccion


**Hallazgos (David):** el supuesto de 3 categorías por cuantiles (33/33/33) no se sostiene con datos reales — `down_up_ratio` está concentrado en 0 (p25=mediana=p66=0.0), así que **p33 = p66 = 0.0**. Eso colapsa las 3 categorías a 2: `balanceado` (down_up_ratio == 0) es el **99.85%** de los flujos (396 747 de 397 354) y `descarga_dominante` solo el 0.15% (607); `carga_dominante` queda **vacía**. El accuracy alto (línea base=0.9958, ganador `RandomForestClassifier`=0.9978, solo +0.20 pp) es un artefacto de ese desbalance, no evidencia de que el modelo prediga bien la dirección dominante: de los 110 flujos reales de `descarga_dominante` en prueba, identificó 51 (~46%). **Recomendación:** separar `down_up_ratio == 0` como categoría propia o recalibrar los cortes solo sobre la subpoblación con `down_up_ratio > 0`, antes de usar esta dimensión como base de U2.

---
## Cierre de fases y alcance

Este notebook cubre, para las 4 dimensiones del equipo, las **5 fases de CRISP-DM hasta el modelado/evaluación**: Comprensión del negocio → Comprensión de los datos → Preparación de los datos → Modelado → Evaluación. **Ninguna incluye la Fase 6 (Despliegue):** poner cada modelo a inferir sobre flujos en vivo es la dimensión U2 de cada integrante, contenido de Unidad 2 (Spark Structured Streaming + Kafka), declarado en el [Brief técnico-analítico](../../docs/proyecto-sello/brief.md).

### Resumen de resultados (ejecutado contra `TRCU.csv`, 397 354 flujos)

| Integrante | Modelo ganador | Métrica principal | vs. línea base | ¿Cumple criterio de éxito? |
|---|---|---|---|---|
| Nick | RandomForestRegressor | R²=0.7784, RMSE=16.1M | RMSE -53% | Sí (R²>0.5) |
| Jhan | RandomForestRegressor | R²=0.8986, RMSE=7.39M | RMSE -70.0% | Sí (≥20%) |
| Henyelrey | RandomForestClassifier | Accuracy=0.9931, F1=0.9927 | +2.4 pp | **No** (objetivo +15-20 pp) |
| David | RandomForestClassifier | Accuracy=0.9978, F1=0.9974 | +0.20 pp | Trivialmente (>50%), pero engañoso — ver hallazgo |

`RandomForestRegressor`/`RandomForestClassifier` ganó las 4 comparaciones frente a los modelos lineales/logísticos (con y sin regularización) — señal consistente de que las relaciones entre las features de un flujo de red y sus 4 variables objetivo son fuertemente no lineales.

### Hallazgos transversales de calidad de datos

- **Duplicados:** entre 66.5% (Nick/Henyelrey/David) y 90% (Jhan, tras filtrar duraciones > 0) de los flujos capturados se eliminan al deduplicar por `flow_id` — señal a investigar en la captura de Suricata antes de escalar a Unidad 2.
- **Catálogo IANA insuficiente (Henyelrey):** solo cubre el 1.7% de los flujos reales; limita el margen de mejora del modelo pese a su alta accuracy nominal.
- **Distribución degenerada de `down_up_ratio` (David):** el diseño de 3 categorías balanceadas por cuantiles no se sostiene frente a los datos reales, fuertemente concentrados en 0.

### Próximos pasos hacia Unidad 2

1. Ampliar el catálogo IANA (Henyelrey) y rediseñar la categorización de dirección dominante (David) antes de calibrar sus modelos U2.
2. Investigar la causa de la alta tasa de duplicados por `flow_id` en la captura de Suricata.
3. Levantar la capa de velocidad (Kafka + Spark Structured Streaming) sobre el mismo esquema de flujos, calibrando cada modelo U2 con el modelo/línea base entrenado en este notebook.
4. Integrar los cuatro paneles U2 en un único tablero Grafana de "salud y comportamiento del tráfico del campus".

Detalle completo por dimensión (glosario propio, EDA específico, features y hallazgos extendidos) en los notebooks individuales (`u1_producto_nick.ipynb`, `u1_producto_jhan.ipynb`, `u1_producto_henyelrey.ipynb`, `u1_producto_david.ipynb`) y en sus [páginas de contribución](../../docs/index.md#contenido-del-sitio), y el proceso metodológico general en [Aplicación de CRISP-DM al proyecto](../../docs/productos/Aplicacion_de_CRISPDM_al_Proyecto.md).